## Exploratory Testing 

In [35]:
# exploratory comparison - Cyclic ID vs Standard ID

# ==================================================
# Compare the cyclic ID (on a cyclic graph) vs standard ID (on an acyclic graph)
# - 20 single-gene perturbations
# - 20 multi-gene perturbations
# Total - 40 comparison runs
# ====================================================

# imports needed
import pandas as pd
import time
from y0.dsl import Variable, P
import networkx as nx
from y0.algorithm.identify import Unidentifiable, identify_outcomes
from y0.algorithm.ioscm.utils import scc_to_bidirected
from y0.graph import NxMixedGraph
from y0.algorithm.identify.utils import Identification



In [36]:
## Load E. coli GRN 

ecoli_graph = nx.read_graphml("ecoli_full_network_no_small_rna.graphml")

print(f"✓ Network loaded (NetworkX):")
print(f"  - Nodes: {ecoli_graph.number_of_nodes():,}")
print(f"  - Edges: {ecoli_graph.number_of_edges():,}")
print(f"  - Is DAG: {nx.is_directed_acyclic_graph(ecoli_graph)}")

# Convert to NxMixedGraph for causal inference
ecoli_graph = NxMixedGraph.from_edges(
    directed=list(ecoli_graph.edges())
)

print(f"\n✓ Converted to NxMixedGraph")
print(f"  - Total nodes: {len(ecoli_graph):,}")
print(f"  - Directed edges: {ecoli_graph.directed.number_of_edges():,}")
print(f"  - Undirected edges: {ecoli_graph.undirected.number_of_edges():,}")



✓ Network loaded (NetworkX):
  - Nodes: 2,976
  - Edges: 9,211
  - Is DAG: False

✓ Converted to NxMixedGraph
  - Total nodes: 2,962
  - Directed edges: 9,211
  - Undirected edges: 0


In [47]:
# pre processing step 

from y0.algorithm.ioscm.utils import scc_to_bidirected
start = time.time()

ecoli_acyclic = scc_to_bidirected(ecoli_graph)

# ecoli_admg = ecoli_acyclic.to_admg()
admg_conversion_time = time.time() - start

print(f"Converted to ADMG:")
print(f"  - Conversion time: {admg_conversion_time:.2f} seconds")
print(f"  - Directed edges: {len(ecoli_acyclic.directed.edges()):,}")
print(f"  - Bidirected edges: {len(ecoli_acyclic.undirected.edges()):,}")
print(f"\n✓ ADMG ready for standard ID algorithm")


Converted to ADMG:
  - Conversion time: 2.83 seconds
  - Directed edges: 8,692
  - Bidirected edges: 484

✓ ADMG ready for standard ID algorithm


In [48]:
# load the test cases from CSV

# single gene perturbations
df_single = pd.read_csv("ecoli_benchmark_all_combined.csv")
print(f"✓ Loaded single-gene data: {len(df_single):,} queries")
print(f"  - Identifiable: {df_single['identifiable'].sum():,}")
print(f"  - Unidentifiable: {(~df_single['identifiable']).sum():,}")

# Multi-gene perturbations
df_multi = pd.read_csv('ecoli_5x5_1000samples_FINAL.csv')
print(f"\n✓ Loaded multi-gene data: {len(df_multi):,} queries")
print(f"  - Identifiable: {df_multi['identifiable'].sum():,}")
print(f"  - Unidentifiable: {(~df_multi['identifiable']).sum():,}")

# Select test queries
print("\n" + "="*60)
print("SELECTING TEST QUERIES")
print("="*60)

# Single-gene: 10 identifiable + 10 unidentifiable
single_identifiable = df_single[df_single['identifiable'] == True].sample(n=10, random_state=42)

# make random state and number of samples constants 

single_unidentifiable = df_single[df_single['identifiable'] == False].sample(n=10, random_state=42)

# Multi-gene: 10 identifiable + 10 unidentifiable
multi_identifiable = df_multi[df_multi['identifiable'] == True].sample(n=10, random_state=42)
multi_unidentifiable = df_multi[df_multi['identifiable'] == False].sample(n=10, random_state=42)

print(f"\n✓ Selected 40 test queries:")
print(f"  - Single-gene identifiable: {len(single_identifiable)}")
print(f"  - Single-gene unidentifiable: {len(single_unidentifiable)}")
print(f"  - Multi-gene identifiable: {len(multi_identifiable)}")
print(f"  - Multi-gene unidentifiable: {len(multi_unidentifiable)}")

✓ Loaded single-gene data: 555 queries
  - Identifiable: 63
  - Unidentifiable: 492

✓ Loaded multi-gene data: 1,000 queries
  - Identifiable: 893
  - Unidentifiable: 107

SELECTING TEST QUERIES

✓ Selected 40 test queries:
  - Single-gene identifiable: 10
  - Single-gene unidentifiable: 10
  - Multi-gene identifiable: 10
  - Multi-gene unidentifiable: 10


In [51]:


def compare_cyclic_vs_standardID(
    admg,                    # y0.NxMixedGraph (ADMG structure, acyclic)
    intervention,            # Gene name(s) - string or list
    target,                  # Gene name(s) - string or list
    cyclic_result_from_csv   # Boolean from CSV
):
    
    
    results = {
        'intervention': str(intervention),
        'target': str(target),
        'cyclic_id_result': cyclic_result_from_csv
    }
    
    # Run standard ID on the ADMG structure
    start = time.time()
    try:
        # Convert to Variable objects
        if isinstance(target, str):
            target_var = {Variable(target)}
        else:
            target_var = {Variable(g.strip()) for g in target}
            
        if isinstance(intervention, str):
            intervention_var = {Variable(intervention)}
        else:
            intervention_var = {Variable(g.strip()) for g in intervention}
        
        # Run standard ID directly - NO CONVERSION NEEDED!
        estimand = identify_outcomes(
            graph=admg,  
            outcomes=target_var,
            treatments=intervention_var
        )
        results['standard_id_result'] = True
        
    except Unidentifiable:
        results['standard_id_result'] = False
    except Exception as e:
        results['standard_id_result'] = False
        results['standard_id_error'] = str(e)[:200]
        
    results['standard_id_time'] = time.time() - start
    
    # Compare results
    results['results_match'] = (
        results['cyclic_id_result'] == results['standard_id_result']
    )
    
    return results

print("✓ Comparison function defined")

✓ Comparison function defined


In [55]:

comparison_results = []

# ============================================================
# [1/4] Single-gene identifiable (10 queries)
# ============================================================
print("\n" + "="*70)
print("[1/4] Running 10 single-gene IDENTIFIABLE queries...")
print("="*70)

for i, (idx, row) in enumerate(single_identifiable.iterrows(), 1):
    print(f"Query {i}/10: {row['intervention']} → {row['target']}", end="...")
    
    result = compare_cyclic_vs_standardID(
        admg=ecoli_acyclic,  # ← y0.NxMixedGraph, no conversion!
        intervention=row['intervention'],
        target=row['target'],
        cyclic_result_from_csv=row['identifiable']
    )
    
    result['query_type'] = 'single_identifiable'
    result['query_number'] = i
    comparison_results.append(result)
    
    print(f" ✓ (std: {result['standard_id_time']:.2f}s)")

# ============================================================
# [2/4] Single-gene unidentifiable (10 queries)
# ============================================================
print("\n" + "="*70)
print("[2/4] Running 10 single-gene UNIDENTIFIABLE queries...")
print("="*70)

for i, (idx, row) in enumerate(single_unidentifiable.iterrows(), 1):
    print(f"Query {i}/10: {row['intervention']} → {row['target']}", end="...")
    
    result = compare_cyclic_vs_standardID(
        admg=ecoli_acyclic,
        intervention=row['intervention'],
        target=row['target'],
        cyclic_result_from_csv=row['identifiable']
    )
    
    result['query_type'] = 'single_unidentifiable'
    result['query_number'] = i
    comparison_results.append(result)
    
    print(f" ✓ (std: {result['standard_id_time']:.2f}s)")

# ============================================================
# [3/4] Multi-gene identifiable (10 queries)
# ============================================================
print("\n" + "="*70)
print("[3/4] Running 10 multi-gene IDENTIFIABLE queries...")
print("="*70)

for i, (idx, row) in enumerate(multi_identifiable.iterrows(), 1):
    interventions = [g.strip() for g in row['interventions'].split(',')]
    outcomes = [g.strip() for g in row['outcomes'].split(',')]
    
    print(f"Query {i}/10: {len(interventions)} genes → {len(outcomes)} genes", end="...")
    
    result = compare_cyclic_vs_standardID(
        admg=ecoli_acyclic,
        intervention=interventions,
        target=outcomes,
        cyclic_result_from_csv=row['identifiable']
    )
    
    result['query_type'] = 'multi_identifiable'
    result['query_number'] = i
    comparison_results.append(result)
    
    print(f" ✓ (std: {result['standard_id_time']:.2f}s)")

# ============================================================
# [4/4] Multi-gene unidentifiable (10 queries)
# ============================================================
print("\n" + "="*70)
print("[4/4] Running 10 multi-gene UNIDENTIFIABLE queries...")
print("="*70)

for i, (idx, row) in enumerate(multi_unidentifiable.iterrows(), 1):
    interventions = [g.strip() for g in row['interventions'].split(',')]
    outcomes = [g.strip() for g in row['outcomes'].split(',')]
    
    print(f"Query {i}/10: {len(interventions)} genes → {len(outcomes)} genes", end="...")
    
    result = compare_cyclic_vs_standardID(
        admg=ecoli_acyclic,
        intervention=interventions,
        target=outcomes,
        cyclic_result_from_csv=row['identifiable']
    )
    
    result['query_type'] = 'multi_unidentifiable'
    result['query_number'] = i
    comparison_results.append(result)
    
    print(f" ✓ (std: {result['standard_id_time']:.2f}s)")

# Summary
print("\n" + "="*70)
print(f"✅ COMPLETED ALL {len(comparison_results)} COMPARISONS")
print("="*70)

single_count = len([r for r in comparison_results if 'single' in r['query_type']])
multi_count = len([r for r in comparison_results if 'multi' in r['query_type']])

print(f"\n Breakdown:")
print(f"   Single-gene queries: {single_count}")
print(f"   Multi-gene queries: {multi_count}")
print(f"   Total: {len(comparison_results)}")

total_time = sum(r['standard_id_time'] for r in comparison_results)
avg_time = total_time / len(comparison_results)

print(f"\n⏱  Timing:")
print(f"   Total Standard ID time: {total_time:.2f}s")
print(f"   Average per query: {avg_time:.3f}s")




[1/4] Running 10 single-gene IDENTIFIABLE queries...
Query 1/10: glrR → eco... ✓ (std: 0.14s)
Query 2/10: rcdA → livF... ✓ (std: 0.13s)
Query 3/10: glaR → yehW... ✓ (std: 0.15s)
Query 4/10: spf → fucK... ✓ (std: 0.54s)
Query 5/10: nsrR → rclB... ✓ (std: 0.23s)
Query 6/10: btsR → cycA... ✓ (std: 0.23s)
Query 7/10: nsrR → ynbB... ✓ (std: 0.20s)
Query 8/10: nsrR → ldtC... ✓ (std: 0.22s)
Query 9/10: btsR → stfQ... ✓ (std: 0.18s)
Query 10/10: glaR → avtA... ✓ (std: 0.16s)

[2/4] Running 10 single-gene UNIDENTIFIABLE queries...
Query 1/10: gadE → yjjY... ✓ (std: 0.38s)
Query 2/10: gadW → mtlR... ✓ (std: 0.13s)
Query 3/10: slyA → speF... ✓ (std: 0.14s)
Query 4/10: fnr → cadC... ✓ (std: 0.16s)
Query 5/10: phoP → fadA... ✓ (std: 0.14s)
Query 6/10: rpoS → pyrD... ✓ (std: 0.13s)
Query 7/10: mlrA → yghG... ✓ (std: 0.13s)
Query 8/10: fnr → yidQ... ✓ (std: 0.35s)
Query 9/10: cytR → livM... ✓ (std: 0.12s)
Query 10/10: nac → sgbE... ✓ (std: 0.16s)

[3/4] Running 10 multi-gene IDENTIFIABLE queries...


In [28]:
# ==================================================
# STEP 6: Single-Gene Results Analysis
# ==================================================

# Filter to single-gene results
single_results_df = pd.DataFrame([r for r in comparison_results if 'single' in r['query_type']])

print("\n" + "="*70)
print("SINGLE-GENE RESULTS - SUMMARY")
print("="*70)

if len(single_results_df) == 0:
    print("\n  No results found. Please run Cell 6 first.")
else:
    # Calculate key metrics
    total = len(single_results_df)
    matches = single_results_df['results_match'].sum()
    match_rate = (matches / total) * 100
    
    # Split by query type
    identifiable_df = single_results_df[single_results_df['query_type'] == 'single_identifiable']
    unidentifiable_df = single_results_df[single_results_df['query_type'] == 'single_unidentifiable']
    
    # Overall summary
    print(f"\n Overall:")
    print(f"   Queries tested: {total}")
    print(f"   Agreement rate: {matches}/{total} ({match_rate:.1f}%)")
    print(f"   Avg time/query: {single_results_df['standard_id_time'].mean():.3f}s")
    
    # Identifiable queries
    if len(identifiable_df) > 0:
        ident_matches = identifiable_df['results_match'].sum()
        print(f"\n✅ IDENTIFIABLE (Cyclic ID = YES):")
        print(f"   Tested: {len(identifiable_df)}")
        print(f"   Standard ID agreed: {ident_matches}/{len(identifiable_df)} ({ident_matches/len(identifiable_df)*100:.1f}%)")
    
    # Unidentifiable queries
    if len(unidentifiable_df) > 0:
        unident_matches = unidentifiable_df['results_match'].sum()
        print(f"\n❌ UNIDENTIFIABLE (Cyclic ID = NO):")
        print(f"   Tested: {len(unidentifiable_df)}")
        print(f"   Standard ID agreed: {unident_matches}/{len(unidentifiable_df)} ({unident_matches/len(unidentifiable_df)*100:.1f}%)")
    
    # Key insight
    print(f"\n💡 KEY INSIGHT:")
    if match_rate == 100:
        print(f"   ✅ Perfect agreement! Standard ID validates Cyclic ID.")
    elif match_rate >= 50:
        # Check the pattern
        ident_agree_pct = (identifiable_df['results_match'].sum() / len(identifiable_df) * 100) if len(identifiable_df) > 0 else 0
        unident_agree_pct = (unidentifiable_df['results_match'].sum() / len(unidentifiable_df) * 100) if len(unidentifiable_df) > 0 else 0
    
    
    # Sample results table
    print(f"\n Sample Results Table:")
    print("-" * 70)
    display(single_results_df[['query_number', 'intervention', 'target', 
                                'cyclic_id_result', 'standard_id_result', 
                                'results_match']].head(5))
    
    # # Save results
    # single_results_df.to_csv('single_gene_comparison_results.csv', index=False)
    # print(f"\n Saved: single_gene_comparison_results.csv")
    
    print("\n" + "="*70)


SINGLE-GENE RESULTS - SUMMARY

 Overall:
   Queries tested: 20
   Agreement rate: 10/20 (50.0%)
   Avg time/query: 0.306s

✅ IDENTIFIABLE (Cyclic ID = Agree(Yes)):
   Tested: 10
   Standard ID agreed: 10/10 (100.0%)

❌ UNIDENTIFIABLE (Cyclic ID does not Agree(No)):
   Tested: 10
   Standard ID agreed: 0/10 (0.0%)

💡 KEY INSIGHT:

 Sample Results Table:
----------------------------------------------------------------------


,query_number,intervention,target,cyclic_id_result,standard_id_result,results_match
0,1,glrR,eco,True,True,True
1,2,rcdA,livF,True,True,True
2,3,glaR,yehW,True,True,True
3,4,spf,fucK,True,True,True
4,5,nsrR,rclB,True,True,True


In [58]:
# analyzing all of the queries (40)
from datetime import datetime
import json

results_df = pd.DataFrame(comparison_results)

print("\n" + "="*70)
print("COMPLETE RESULTS - CYCLIC ID vs STANDARD ID")
print("="*70)

# Overall summary
total = len(results_df)
matches = results_df['results_match'].sum()
match_rate = (matches / total) * 100

print(f"\n📊 OVERALL (All 40 queries):")
print(f"   Queries tested: {total}")
print(f"   Agreement rate: {matches}/{total} ({match_rate:.1f}%)")
print(f"   Disagreements: {total - matches}")
print(f"   Avg time/query: {results_df['standard_id_time'].mean():.3f}s")
print(f"   Total time: {results_df['standard_id_time'].sum():.2f}s")

# ============================================================
# Breakdown by Perturbation Type (Single vs Multi)
# ============================================================
print(f"\n" + "-"*70)
print("📋 BY PERTURBATION TYPE:")
print("-"*70)

single_df = results_df[results_df['query_type'].str.contains('single')]
multi_df = results_df[results_df['query_type'].str.contains('multi')]

print(f"\n🔹 SINGLE-GENE Perturbations:")
print(f"   Queries: {len(single_df)}")
print(f"   Agreement: {single_df['results_match'].sum()}/{len(single_df)} ({single_df['results_match'].sum()/len(single_df)*100:.1f}%)")
print(f"   Avg time: {single_df['standard_id_time'].mean():.3f}s")

print(f"\n🔸 MULTI-GENE Perturbations:")
print(f"   Queries: {len(multi_df)}")
print(f"   Agreement: {multi_df['results_match'].sum()}/{len(multi_df)} ({multi_df['results_match'].sum()/len(multi_df)*100:.1f}%)")
print(f"   Avg time: {multi_df['standard_id_time'].mean():.3f}s")

# ============================================================
# Breakdown by All 4 Categories
# ============================================================
print(f"\n" + "-"*70)
print("📋 BY CATEGORY (Detailed):")
print("-"*70)

categories = [
    ('single_identifiable', '✅ Single-gene IDENTIFIABLE'),
    ('single_unidentifiable', '❌ Single-gene UNIDENTIFIABLE'),
    ('multi_identifiable', '✅ Multi-gene IDENTIFIABLE'),
    ('multi_unidentifiable', '❌ Multi-gene UNIDENTIFIABLE')
]

for cat_name, cat_label in categories:
    cat_df = results_df[results_df['query_type'] == cat_name]
    if len(cat_df) > 0:
        matches_cat = cat_df['results_match'].sum()
        total_cat = len(cat_df)
        rate = (matches_cat / total_cat * 100)
        
        print(f"\n{cat_label}:")
        print(f"   Tested: {total_cat}")
        print(f"   Standard ID agreed: {matches_cat}/{total_cat} ({rate:.1f}%)")
        print(f"   Avg time: {cat_df['standard_id_time'].mean():.3f}s")

# ============================================================
# Pattern Analysis
# ============================================================
print(f"\n" + "-"*70)
print("💡 KEY PATTERNS:")
print("-"*70)

# Calculate agreement rates for identifiable vs unidentifiable
ident_df = results_df[results_df['query_type'].str.contains('identifiable') & 
                      ~results_df['query_type'].str.contains('unidentifiable')]
unident_df = results_df[results_df['query_type'].str.contains('unidentifiable')]

ident_agree = ident_df['results_match'].sum()
ident_total = len(ident_df)
unident_agree = unident_df['results_match'].sum()
unident_total = len(unident_df)

ident_agree_rate = (ident_agree / ident_total * 100) if ident_total > 0 else 0
unident_agree_rate = (unident_agree / unident_total * 100) if unident_total > 0 else 0

print(f"\n When Cyclic ID says IDENTIFIABLE ({ident_total} queries):")
print(f"   Standard ID agreed: {ident_agree}/{ident_total} ({ident_agree_rate:.1f}%)")

print(f"\n When Cyclic ID says UNIDENTIFIABLE ({unident_total} queries):")
print(f"   Standard ID agreed: {unident_agree}/{unident_total} ({unident_agree_rate:.1f}%)")



# Check for single vs multi differences
if len(single_df) > 0 and len(multi_df) > 0:
    single_match_rate = (single_df['results_match'].sum() / len(single_df) * 100)
    multi_match_rate = (multi_df['results_match'].sum() / len(multi_df) * 100)
    
    if abs(single_match_rate - multi_match_rate) > 10:
        print(f"\n PERTURBATION TYPE DIFFERENCE:")
        print(f"   → Single-gene: {single_match_rate:.1f}% agreement")
        print(f"   → Multi-gene: {multi_match_rate:.1f}% agreement")
        if multi_match_rate > single_match_rate:
            print(f"   → Multi-gene perturbations show BETTER agreement")
        else:
            print(f"   → Single-gene perturbations show BETTER agreement")

# ============================================================
# Sample Results Table
# ============================================================
print(f"\n" + "-"*70)
print("📋 SAMPLE RESULTS (3 from each category):")
print("-"*70)

for cat_name, cat_label in categories:
    cat_df = results_df[results_df['query_type'] == cat_name]
    if len(cat_df) > 0:
        print(f"\n{cat_label}:")
        display(cat_df[['query_number', 'intervention', 'target', 
                        'cyclic_id_result', 'standard_id_result', 
                        'results_match', 'standard_id_time']].head(3))

# ============================================================
# Save Results
# ============================================================
print(f"\n" + "="*70)

# Save complete results
results_df.to_csv('complete_comparison_results.csv', index=False)
print(f"💾 Complete results saved: complete_comparison_results.csv")

# Save summary statistics
summary_stats = {
    'total_queries': int(total),
    'agreements': int(matches),
    'disagreements': int(total - matches),
    'agreement_rate_percent': float(match_rate),
    'single_gene_queries': int(len(single_df)),
    'multi_gene_queries': int(len(multi_df)),
    'single_gene_agreement_rate': float(single_df['results_match'].sum()/len(single_df)*100) if len(single_df) > 0 else 0,
    'multi_gene_agreement_rate': float(multi_df['results_match'].sum()/len(multi_df)*100) if len(multi_df) > 0 else 0,
    'identifiable_agreement_rate': float(ident_agree_rate),
    'unidentifiable_agreement_rate': float(unident_agree_rate),
    'mean_time_seconds': float(results_df['standard_id_time'].mean()),
    'total_time_seconds': float(results_df['standard_id_time'].sum()),
    'analysis_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

with open('comparison_summary.json', 'w') as f:
    json.dump(summary_stats, f, indent=2)
print(f"💾 Summary statistics saved: comparison_summary.json")

print(f"\n" + "="*70)
print("✅ Analysis Complete!")
print("="*70)

print(f"\n📌 KEY TAKEAWAY:")
print(f"   Tested {total} queries ({len(single_df)} single-gene, {len(multi_df)} multi-gene)")
print(f"   Overall agreement: {match_rate:.1f}%")



COMPLETE RESULTS - CYCLIC ID vs STANDARD ID

📊 OVERALL (All 40 queries):
   Queries tested: 40
   Agreement rate: 20/40 (50.0%)
   Disagreements: 20
   Avg time/query: 0.189s
   Total time: 7.56s

----------------------------------------------------------------------
📋 BY PERTURBATION TYPE:
----------------------------------------------------------------------

🔹 SINGLE-GENE Perturbations:
   Queries: 20
   Agreement: 10/20 (50.0%)
   Avg time: 0.200s

🔸 MULTI-GENE Perturbations:
   Queries: 20
   Agreement: 10/20 (50.0%)
   Avg time: 0.178s

----------------------------------------------------------------------
📋 BY CATEGORY (Detailed):
----------------------------------------------------------------------

✅ Single-gene IDENTIFIABLE:
   Tested: 10
   Standard ID agreed: 10/10 (100.0%)
   Avg time: 0.217s

❌ Single-gene UNIDENTIFIABLE:
   Tested: 10
   Standard ID agreed: 0/10 (0.0%)
   Avg time: 0.183s

✅ Multi-gene IDENTIFIABLE:
   Tested: 10
   Standard ID agreed: 10/10 (100.0%)
 

,query_number,intervention,target,cyclic_id_result,standard_id_result,results_match,standard_id_time
0,1,glrR,eco,True,True,True,0.139429
1,2,rcdA,livF,True,True,True,0.130421
2,3,glaR,yehW,True,True,True,0.153642



❌ Single-gene UNIDENTIFIABLE:


,query_number,intervention,target,cyclic_id_result,standard_id_result,results_match,standard_id_time
10,1,gadE,yjjY,False,True,False,0.376760
11,2,gadW,mtlR,False,True,False,0.131110
12,3,slyA,speF,False,True,False,0.137141



✅ Multi-gene IDENTIFIABLE:


,query_number,intervention,target,cyclic_id_result,standard_id_result,results_match,standard_id_time
20,1,"['ycaM', 'tufB', 'ydhV', 'hslO', 'blr']","['atoC', 'cysH', 'chpS', 'pitA', 'yqeH']",True,True,True,0.151213
21,2,"['mdtF', 'nuoK', 'pgk', 'fimB', 'lnt']","['ydeO', 'lacZ', 'pyrC', 'yihM', 'sthA']",True,True,True,0.141253
22,3,"['pabA', 'rrsC', 'livJ', 'ybeZ', 'ytfP']","['ptrA', 'iscS', 'argU', 'slyB', 'waaF']",True,True,True,0.133284



❌ Multi-gene UNIDENTIFIABLE:


,query_number,intervention,target,cyclic_id_result,standard_id_result,results_match,standard_id_time
30,1,"['patD', 'ydiL', 'pdeB', 'paaX', 'rstA']","['yagE', 'malF', 'frmR', 'thiQ', 'ubiU']",False,True,False,0.159884
31,2,"['qseB', 'ydfX', 'ybiX', 'rutR', 'rpiA']","['tauA', 'thrL', 'ileV', 'xapB', 'melB']",False,True,False,0.364853
32,3,"['lon', 'ydgI', 'cysI', 'cbrC', 'ihfA']","['ykgO', 'fadA', 'greB', 'erpA', 'narJ']",False,True,False,0.269002



💾 Complete results saved: complete_comparison_results.csv
💾 Summary statistics saved: comparison_summary.json

✅ Analysis Complete!

📌 KEY TAKEAWAY:
   Tested 40 queries (20 single-gene, 20 multi-gene)
   Overall agreement: 50.0%
